# PesoLens retrain — imgsz 256 (latency reduction), v2

Same data, class remap, and hyperparameters as the client-approved **thesis2** model
(2026-07-07, imgsz 320, Ultralytics 8.4.90). **Only `IMG_SIZE` changes (320 -> 256).**

Post-training cells never raise (a failed Kaggle run discards all outputs). The TFLite export
runs under **Ultralytics 8.4.76**, because >=8.4.83 exports LiteRT float32-NCHW models the
firmware cannot use. Final artifact: `/kaggle/working/pesolens_img256_full.zip`.

In [ ]:
# Preflight: fail in the first minute if Kaggle silently dropped GPU / internet / dataset.
import subprocess, socket, torch
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'NO GPU - set Accelerator to GPU T4 x2'
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
socket.create_connection(('pypi.org', 443), timeout=10).close()
print('Internet: OK')
from pathlib import Path
assert any(Path('/kaggle/input').rglob('data.yaml')), 'Dataset not attached (pesolens-july7)'
print('Dataset: attached')

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Training uses the version that trained the client-approved thesis2 model (2026-07-07).
!pip install -q ultralytics==8.4.90 pillow-heif

In [ ]:
from pathlib import Path
from collections import Counter
import shutil, zipfile
from PIL import Image
from pillow_heif import register_heif_opener
register_heif_opener()

# Firmware class order — DO NOT reorder.
TARGET = ['20', '50', '100', '500', '1000']
target_index = {d: i for i, d in enumerate(TARGET)}

# Roboflow 9-class index -> denomination string
NINE_TO_DENOM = {
    0: '100', 1: '1000', 2: '50', 3: '500',
    4: '100', 5: '1000', 6: '20', 7: '50', 8: '500',
}
REMAP = {i: target_index[NINE_TO_DENOM[i]] for i in range(9)}
print('9->5 label remap:', REMAP)

OUT = Path('/kaggle/working/pesolens_merged')
IMG_SIZE = 256  # was 320 in the approved model; the ONLY change

In [ ]:
from pathlib import Path
print('=== top of /kaggle/input ===')

for p in sorted(Path('/kaggle/input').rglob('*')):
    if p.is_dir():
        print('DIR ', p)
print('\n=== data.yaml files ===')
for y in Path('/kaggle/input').rglob('data.yaml'):
    print(' ', y)
print('\n=== .zip files ===')
for z in Path('/kaggle/input').rglob('*.zip'):
    print(' ', z)

In [ ]:
# Kaggle auto-extracts uploaded zips, so locate each dataset by its data.yaml
yamls = sorted(Path('/kaggle/input').rglob('data.yaml'))
print('Found dataset roots:')
for y in yamls:
    print(' ', y.parent)
roots = [y.parent for y in yamls]
assert len(roots) == 2, f'Expected 2 dataset roots, found {len(roots)}: {roots}'

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
for split in ['train', 'valid', 'test']:
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.heic', '.heif'}

def remap_label(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        parts[0] = str(REMAP[int(float(parts[0]))])
        out.append(' '.join(parts))
    return ('\n'.join(out) + '\n') if out else ''

for zi, root in enumerate(roots):
    prefix = f'z{zi}'
    for split in ['train', 'valid', 'test']:
        idir = root / split / 'images'
        ldir = root / split / 'labels'
        if not idir.exists():
            continue
        for img in idir.iterdir():
            if img.suffix.lower() not in IMG_EXT:
                continue
            stem = img.stem
            if img.suffix.lower() in {'.heic', '.heif'}:
                dst_img = OUT / split / 'images' / f'{prefix}_{stem}.jpg'
                Image.open(img).convert('RGB').save(dst_img, quality=92)
            else:
                dst_img = OUT / split / 'images' / f'{prefix}_{stem}{img.suffix.lower()}'
                shutil.copy2(img, dst_img)
            lbl = ldir / f'{stem}.txt'
            dst_lbl = OUT / split / 'labels' / f'{prefix}_{stem}.txt'
            dst_lbl.write_text(remap_label(lbl.read_text()) if lbl.exists() else '')

print('Merge done.')

In [ ]:
EXPECTED_IMAGES = {'train': 3891, 'valid': 567, 'test': 566}
EXPECTED_HIST = Counter({0: 644, 1: 1675, 2: 1582, 3: 1377, 4: 1374})

hist = Counter()
for split in ['train', 'valid', 'test']:
    imgs = list((OUT / split / 'images').glob('*'))
    lbls = list((OUT / split / 'labels').glob('*.txt'))
    print(f'{split:5s}: {len(imgs)} imgs, {len(lbls)} labels')
    assert len(imgs) == EXPECTED_IMAGES[split], (split, len(imgs))
    for lp in lbls:
        for line in lp.read_text().splitlines():
            if line.strip():
                hist[int(line.split()[0])] += 1

print('Class histogram:', {TARGET[k]: hist[k] for k in sorted(hist)})
assert hist == EXPECTED_HIST, (hist, EXPECTED_HIST)
print('AUDIT PASSED — safe to train.')

In [ ]:
import yaml
data = {
    'path': str(OUT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(TARGET),
    'names': TARGET,
}
(OUT / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False))
print((OUT / 'data.yaml').read_text())

In [ ]:
import random
import cv2
from IPython.display import display

def draw(img_path, lbl_path):
    im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for line in Path(lbl_path).read_text().splitlines():
        if not line.strip():
            continue
        c, x, y, bw, bh = line.split()[:5]
        c = int(c); x=float(x)*w; y=float(y)*h; bw=float(bw)*w; bh=float(bh)*h
        p1 = (int(x-bw/2), int(y-bh/2)); p2 = (int(x+bw/2), int(y+bh/2))
        cv2.rectangle(im, p1, p2, (255,0,0), 3)
        cv2.putText(im, f'P{TARGET[c]}', (p1[0], max(0,p1[1]-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,0,0), 3)
    return Image.fromarray(im).resize((400, 300))

samples = list((OUT/'train'/'images').glob('*'))
for img in random.sample(samples, 8):
    lbl = OUT/'train'/'labels'/f'{img.stem}.txt'
    display(draw(img, lbl))
    print(img.name, '->', lbl.read_text().strip()[:80])

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(OUT / 'data.yaml'),
    imgsz=IMG_SIZE,
    epochs=100,
    patience=100,
    batch=16,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='pesolens_5class_yolov8n_img256_thesis2',
    # augmentation — uncontrolled bill orientation on ESP32 camera
    fliplr=0.5,
    flipud=0.5,
    degrees=180.0,
    translate=0.1,
    scale=0.5,
    hsv_h=0.01,
    hsv_s=0.3, hsv_v=0.3,
    mosaic=1.0, close_mosaic=10,
)
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.is_file(), best_pt
print('RUN:', run_dir)
print('BEST:', best_pt)

In [ ]:
# From here on NOTHING may raise: if the notebook errors, Kaggle discards every output file.
import traceback, json
REPORT = {}
def safe(name, fn):
    try:
        REPORT[name] = {'ok': True, 'result': fn()}
    except Exception as e:
        traceback.print_exc()
        REPORT[name] = {'ok': False, 'error': f'{type(e).__name__}: {e}'}
    print(f'[{name}]', 'OK' if REPORT[name]['ok'] else 'FAILED -> ' + REPORT[name]['error'])

In [ ]:
import yaml, ultralytics
def _args():
    APPROVED = dict(model='yolov8n.pt', epochs=100, patience=100, batch=16, workers=2,
                    optimizer='auto', seed=0, lr0=0.01, lrf=0.01,
                    hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, degrees=180.0, translate=0.1,
                    scale=0.5, shear=0.0, perspective=0.0, flipud=0.5, fliplr=0.5,
                    mosaic=1.0, close_mosaic=10, mixup=0.0, cutmix=0.0, copy_paste=0.0)
    args = yaml.safe_load((run_dir / 'args.yaml').read_text())
    diff = {k: (args.get(k), v) for k, v in APPROVED.items() if args.get(k) != v}
    if diff or args['imgsz'] != 256:
        raise ValueError(f'args drifted from approved run: {diff}, imgsz={args["imgsz"]}')
    return {'ultralytics': ultralytics.__version__, 'imgsz': args['imgsz'], 'epochs': args['epochs']}
safe('args_match_approved', _args)

In [ ]:
def _val():
    m = YOLO(str(best_pt)).val(data=str(OUT / 'data.yaml'), imgsz=IMG_SIZE, split='val', plots=False)
    per_class = {TARGET[i]: {'P': float(m.box.p[j]), 'R': float(m.box.r[j]),
                             'mAP50': float(m.box.ap50[j]), 'mAP50-95': float(m.box.ap[j])}
                 for j, i in enumerate(m.box.ap_class_index)}
    metrics = {'imgsz': IMG_SIZE, 'mAP50': float(m.box.map50), 'mAP50-95': float(m.box.map),
               'precision': float(m.box.mp), 'recall': float(m.box.mr), 'per_class': per_class}
    (run_dir / 'val_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))
    return {'mAP50': metrics['mAP50'], 'mAP50-95': metrics['mAP50-95']}
safe('validate', _val)

In [ ]:
# Export with Ultralytics 8.4.76 (the version that produced the model on the board), isolated
# in a subprocess so the 8.4.90 training environment is untouched.
import subprocess, sys, os, shutil
EXP_DIR = Path('/kaggle/working/export_ul8476')
UL = '/kaggle/tmp/ul8476'
def _export():
    EXP_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_pt, EXP_DIR / 'best.pt')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', UL,
                        'ultralytics==8.4.76'], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    script = (
        "import ultralytics; print('export with ultralytics', ultralytics.__version__)\n"
        "assert ultralytics.__version__ == '8.4.76', ultralytics.__version__\n"
        "from ultralytics import YOLO\n"
        f"p = YOLO(r'{EXP_DIR / 'best.pt'}').export(format='tflite', int8=True, imgsz={IMG_SIZE}, "
        f"data=r'{OUT / 'data.yaml'}')\n"
        "print('EXPORTED', p)\n")
    env = dict(os.environ, PYTHONPATH=UL)
    r = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True, env=env,
                       cwd=str(EXP_DIR), timeout=3600)
    print(r.stdout[-8000:]); print(r.stderr[-8000:])
    if r.returncode != 0:
        raise RuntimeError(f'export subprocess exit {r.returncode}')
    return [str(p) for p in EXP_DIR.rglob('*.tflite')]
safe('export_ul8476', _export)

In [ ]:
# Firmware contract (firmware/src/inference.cpp): input int8 [1,256,256,3] scale 1/255 zp -128,
# output int8 [1,9,1344], every op registered in the MicroMutableOpResolver.
import numpy as np, collections
FW_OPS = {'ADD','CONCATENATION','CONV_2D','DEPTHWISE_CONV_2D','DEQUANTIZE','FULLY_CONNECTED',
          'HARD_SWISH','LEAKY_RELU','LOGISTIC','MAX_POOL_2D','MEAN','MUL','PAD','QUANTIZE',
          'RESHAPE','RESIZE_BILINEAR','RESIZE_NEAREST_NEIGHBOR','SHAPE','SLICE','SOFTMAX',
          'SPLIT','SPLIT_V','SQUEEZE','STRIDED_SLICE','SUB','TRANSPOSE','UNPACK'}
A = (IMG_SIZE//8)**2 + (IMG_SIZE//16)**2 + (IMG_SIZE//32)**2
def _check_one(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf; Interpreter = tf.lite.Interpreter
    it = Interpreter(model_path=str(path)); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    ops = collections.Counter(d['op_name'] for d in it._get_ops_details())
    problems = []
    if i['dtype'] != np.int8: problems.append(f"input dtype {i['dtype'].__name__}")
    if [int(x) for x in i['shape']] != [1, IMG_SIZE, IMG_SIZE, 3]: problems.append(f"input shape {[int(x) for x in i['shape']]}")
    s, z = i['quantization']
    if not (abs(s - 1/255) < 1e-6 and z == -128): problems.append(f'input quant ({s}, {z})')
    if o['dtype'] != np.int8: problems.append(f"output dtype {o['dtype'].__name__}")
    if sorted(int(x) for x in o['shape'][1:]) != sorted([9, A]): problems.append(f"output shape {[int(x) for x in o['shape']]}")
    extra = set(ops) - FW_OPS - {'DELEGATE'}
    if extra: problems.append(f'ops not in firmware resolver: {sorted(extra)}')
    return {'problems': problems, 'input': [[int(x) for x in i['shape']], [float(s), int(z)]],
            'output': [int(x) for x in o['shape']], 'ops': dict(ops)}
def _contract():
    res = {}
    for p in sorted(EXP_DIR.rglob('*.tflite')):
        try:
            res[str(p.relative_to(EXP_DIR))] = _check_one(p)
        except Exception as e:
            res[str(p.relative_to(EXP_DIR))] = {'problems': [f'could not load: {e}']}
    for k, v in res.items():
        print(('OK   ' if not v['problems'] else 'FAIL ') + k, v['problems'] or '')
    passing = [k for k, v in res.items() if not v['problems']]
    (EXP_DIR / 'contract_check.json').write_text(json.dumps(res, indent=2))
    if passing:
        shutil.copy2(EXP_DIR / passing[0], EXP_DIR / 'FIRMWARE_READY_best_full_integer_quant.tflite')
    return {'passing': passing}
safe('firmware_contract', _contract)

In [ ]:
# Package EVERYTHING (weights, confusion matrices, curves, results.csv, args.yaml, exports),
# then drop the merged dataset copy so the saved output stays small.
import hashlib
def _package():
    pkg = Path('/kaggle/working/pesolens_img256_full')
    if pkg.exists(): shutil.rmtree(pkg)
    shutil.copytree(run_dir, pkg / 'run')
    if EXP_DIR.exists():
        shutil.copytree(EXP_DIR, pkg / 'export_ul8476', ignore=shutil.ignore_patterns('*.onnx'))
    shutil.copy2(OUT / 'data.yaml', pkg / 'data.yaml')
    (pkg / 'REPORT.json').write_text(json.dumps(REPORT, indent=2, default=str))
    z = shutil.make_archive('/kaggle/working/pesolens_img256_full', 'zip', pkg)
    print('ZIP:', z, round(Path(z).stat().st_size / 1e6, 1), 'MB')
    return z
safe('package', _package)
safe('cleanup_merged_dataset', lambda: shutil.rmtree(OUT) or 'removed')
print(json.dumps(REPORT, indent=2, default=str))